# Gaussian Limit Empirical estimation

In [ ]:
from __future__ import annotations

import os
import glob
from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


OBSERVABLES = ["theta1", "w1", "theta2", "w2"]
RESULT_COLS = ["W1_1", "W1_2", "W1_3", "W1_4"]


@dataclass
class Config:
    burn_in: int = 0
    grid_size: int = 80
    lag_cutoff: int = 50
    n_gaussian_sims: int = 5000
    quantile_low: float = 0.01
    quantile_high: float = 0.99
    use_bartlett: bool = True
    random_seed: int = 123
    figure_dpi: int = 180
    bins: int = 40


# -----------------------------
# I/O helpers
# -----------------------------
def find_csvs(folder: str) -> List[str]:
    files = sorted(glob.glob(os.path.join(folder, "*.csv")))
    if not files:
        raise FileNotFoundError(f"No CSV files found in {folder!r}")
    return files


def load_single_path_csv(file_path: str) -> pd.DataFrame:
    df = pd.read_csv(file_path)
    if df.shape[1] < 4:
        raise ValueError(f"Expected at least 4 columns in {file_path}, got {df.shape[1]}")

    cols = list(df.columns[:4])
    rename_map = {cols[0]: "theta1", cols[1]: "w1", cols[2]: "theta2", cols[3]: "w2"}
    df = df.rename(columns=rename_map)
    return df[["theta1", "w1", "theta2", "w2"]].copy()


def load_ensemble(folder: str, burn_in: int = 0) -> Dict[str, List[np.ndarray]]:
    files = find_csvs(folder)
    out = {obs: [] for obs in OBSERVABLES}

    for fp in files:
        df = load_single_path_csv(fp)
        if burn_in > 0:
            df = df.iloc[burn_in:].reset_index(drop=True)
        for obs in OBSERVABLES:
            out[obs].append(df[obs].to_numpy(dtype=float))

    return out


def load_wasserstein_results(results_csv: str) -> pd.DataFrame:
    df = pd.read_csv(results_csv)
    needed = ["file_name", *RESULT_COLS]
    missing = [c for c in needed if c not in df.columns]
    if missing:
        raise ValueError(f"Missing expected columns in {results_csv}: {missing}")
    return df[needed].copy()


# -----------------------------
# Empirical-process covariance estimation
# -----------------------------
def pooled_quantile_grid(paths: List[np.ndarray], grid_size: int, q_low: float, q_high: float) -> np.ndarray:
    pooled = np.concatenate(paths)
    probs = np.linspace(q_low, q_high, grid_size)
    grid = np.quantile(pooled, probs)
    grid = np.unique(grid)
    if grid.size < 5:
        raise ValueError("Grid collapsed to too few unique points; increase variability or grid design.")
    return grid.astype(float)


def empirical_cdf_on_grid(paths: List[np.ndarray], grid: np.ndarray) -> np.ndarray:
    pooled = np.sort(np.concatenate(paths))
    counts = np.searchsorted(pooled, grid, side="right")
    return counts / pooled.size


def centered_indicator_matrix(x: np.ndarray, grid: np.ndarray, Fhat: np.ndarray) -> np.ndarray:
    return (x[:, None] <= grid[None, :]).astype(float) - Fhat[None, :]


def bartlett_weight(k: int, L: int) -> float:
    if L <= 0:
        return 1.0 if k == 0 else 0.0
    val = 1.0 - abs(k) / (L + 1.0)
    return max(val, 0.0)


def estimate_long_run_covariance(paths: List[np.ndarray], grid: np.ndarray, Fhat: np.ndarray, L: int) -> np.ndarray:
    J = grid.size
    Sigma = np.zeros((J, J), dtype=float)
    M = len(paths)

    for x in paths:
        Y = centered_indicator_matrix(x, grid, Fhat)
        n = Y.shape[0]
        if n <= L + 1:
            raise ValueError(f"Path length {n} is too short for lag cutoff {L}.")

        Gamma0 = (Y.T @ Y) / n
        Sigma += Gamma0

        for k in range(1, L + 1):
            wk = bartlett_weight(k, L)
            Gk = (Y[:-k].T @ Y[k:]) / (n - k)
            Sigma += wk * (Gk + Gk.T)

    Sigma /= M
    return Sigma


def nearest_psd(A: np.ndarray, eps: float = 1e-10) -> np.ndarray:
    A = 0.5 * (A + A.T)
    vals, vecs = np.linalg.eigh(A)
    vals = np.clip(vals, eps, None)
    return (vecs * vals) @ vecs.T


# -----------------------------
# Gaussian simulation and integral functional
# -----------------------------
def simulate_gaussian_integrals(grid: np.ndarray, Sigma: np.ndarray, n_sims: int, seed: int) -> np.ndarray:
    rng = np.random.default_rng(seed)
    mean = np.zeros(grid.size, dtype=float)
    Z = rng.multivariate_normal(mean=mean, cov=Sigma, size=n_sims, check_valid="warn")

    vals = np.trapz(np.abs(Z), x=grid, axis=1)
    return vals


# -----------------------------
# Diagnostics / scaling
# -----------------------------
def effective_q_from_paths(paths: List[np.ndarray]) -> int:
    return int(min(len(x) for x in paths))


def scaled_empirical_wasserstein(results_df: pd.DataFrame, obs_idx: int, q: int) -> np.ndarray:
    w = results_df[RESULT_COLS[obs_idx]].to_numpy(dtype=float)
    return np.sqrt(q) * w


def summarize_distribution(x: np.ndarray) -> Dict[str, float]:
    return {
        "mean": float(np.mean(x)),
        "std": float(np.std(x, ddof=1)),
        "q05": float(np.quantile(x, 0.05)),
        "q50": float(np.quantile(x, 0.50)),
        "q95": float(np.quantile(x, 0.95)),
    }


# -----------------------------
# Plotting
# -----------------------------
def plot_system_comparison(
    system_name: str,
    empirical_scaled: Dict[str, np.ndarray],
    predicted: Dict[str, np.ndarray],
    out_png: str,
    bins: int = 40,
    dpi: int = 180,
) -> None:
    fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
    axes = axes.ravel()

    for j, obs in enumerate(OBSERVABLES):
        ax = axes[j]
        emp = empirical_scaled[obs]
        pred = predicted[obs]

        lo = min(emp.min(), pred.min())
        hi = max(emp.max(), pred.max())
        edges = np.linspace(lo, hi, bins + 1)

        ax.hist(emp, bins=edges, density=True, alpha=0.45, label=r"Empirical $\sqrt{q}W$")
        ax.hist(pred, bins=edges, density=True, alpha=0.45, label=r"Predicted $\int |G(t)|dt$")
        ax.set_title(obs)
        ax.set_xlabel("value")
        ax.set_ylabel("density")
        ax.legend(frameon=False)

    fig.suptitle(system_name)
    fig.savefig(out_png, dpi=dpi, bbox_inches="tight")
    plt.close(fig)


# -----------------------------
# End-to-end run
# -----------------------------
def analyze_system(
    paths_folder: str,
    results_csv: str,
    out_dir: str,
    cfg: Config,
) -> Tuple[pd.DataFrame, Dict[str, np.ndarray], Dict[str, np.ndarray]]:
    os.makedirs(out_dir, exist_ok=True)

    ensemble = load_ensemble(paths_folder, burn_in=cfg.burn_in)
    results_df = load_wasserstein_results(results_csv)

    predicted: Dict[str, np.ndarray] = {}
    empirical_scaled: Dict[str, np.ndarray] = {}
    rows = []

    for j, obs in enumerate(OBSERVABLES):
        paths = ensemble[obs]
        grid = pooled_quantile_grid(paths, cfg.grid_size, cfg.quantile_low, cfg.quantile_high)
        Fhat = empirical_cdf_on_grid(paths, grid)
        Sigma = estimate_long_run_covariance(paths, grid, Fhat, cfg.lag_cutoff)
        Sigma_psd = nearest_psd(Sigma)
        sims = simulate_gaussian_integrals(grid, Sigma_psd, cfg.n_gaussian_sims, cfg.random_seed + j)

        q = effective_q_from_paths(paths)
        emp_scaled = scaled_empirical_wasserstein(results_df, j, q)

        predicted[obs] = sims
        empirical_scaled[obs] = emp_scaled

        np.save(os.path.join(out_dir, f"{obs}_grid.npy"), grid)
        np.save(os.path.join(out_dir, f"{obs}_Fhat.npy"), Fhat)
        np.save(os.path.join(out_dir, f"{obs}_Sigma.npy"), Sigma_psd)
        np.save(os.path.join(out_dir, f"{obs}_predicted_integrals.npy"), sims)
        np.save(os.path.join(out_dir, f"{obs}_empirical_scaled_w.npy"), emp_scaled)

        row = {
            "observable": obs,
            "q_used": q,
            "grid_size": int(grid.size),
            "lag_cutoff": cfg.lag_cutoff,
        }
        row.update({f"emp_{k}": v for k, v in summarize_distribution(emp_scaled).items()})
        row.update({f"pred_{k}": v for k, v in summarize_distribution(sims).items()})
        rows.append(row)

    summary = pd.DataFrame(rows)
    summary.to_csv(os.path.join(out_dir, "summary.csv"), index=False)

    system_name = os.path.basename(os.path.normpath(paths_folder))
    plot_system_comparison(
        system_name=system_name,
        empirical_scaled=empirical_scaled,
        predicted=predicted,
        out_png=os.path.join(out_dir, f"{system_name}_comparison.png"),
        bins=cfg.bins,
        dpi=cfg.figure_dpi,
    )

    return summary, empirical_scaled, predicted


def main() -> None:
    cfg = Config(
        burn_in=0,
        grid_size=80,
        lag_cutoff=50,
        n_gaussian_sims=5000,
        quantile_low=0.01,
        quantile_high=0.99,
        random_seed=123,
        figure_dpi=180,
        bins=40,
    )

    jobs = [
        {
            "paths_folder": "paths1",
            "results_csv": os.path.join("AVMs2", "paths1_results.csv"),
            "out_dir": os.path.join("gaussian_empirical_process_output", "paths1"),
        },
        {
            "paths_folder": "paths2",
            "results_csv": os.path.join("AVMs2", "paths2_results.csv"),
            "out_dir": os.path.join("gaussian_empirical_process_output", "paths2"),
        },
    ]

    all_summaries = []
    for job in jobs:
        summary, _, _ = analyze_system(
            paths_folder=job["paths_folder"],
            results_csv=job["results_csv"],
            out_dir=job["out_dir"],
            cfg=cfg,
        )
        summary.insert(0, "system", os.path.basename(job["paths_folder"]))
        all_summaries.append(summary)

    combined = pd.concat(all_summaries, ignore_index=True)
    os.makedirs("gaussian_empirical_process_output", exist_ok=True)
    combined.to_csv(os.path.join("gaussian_empirical_process_output", "combined_summary.csv"), index=False)
    print(combined)


if __name__ == "__main__":
    main()


   system observable  q_used  grid_size  lag_cutoff     emp_mean     emp_std  \
0  paths1     theta1  100002         80          50     4.951879    1.836066   
1  paths1         w1  100002         80          50     8.290468    3.572253   
2  paths1     theta2  100002         80          50     8.690593    3.550735   
3  paths1         w2  100002         80          50    43.826021   25.300633   
4  paths2     theta1  100002         80          50    19.296274   25.793873   
5  paths2         w1  100002         80          50   781.192015  248.007217   
6  paths2     theta2  100002         80          50    11.325602    8.193375   
7  paths2         w2  100002         80          50  1091.023316  342.447115   

      emp_q05      emp_q50      emp_q95  pred_mean   pred_std   pred_q05  \
0    2.599272     4.554505     8.009414   8.172199   3.995733   3.387865   
1    4.063483     7.648517    15.031338  14.828785   7.893322   5.403912   
2    3.833502     8.282209    15.378897  10.389185 